### Welcome to Week 5 Day 1

AutoGen AgentChat!

This should look simple and familiar, because it has a lot in common with Crew and OpenAI Agents SDK

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

### First concept: the Model

In [5]:
from autogen_ext.models.openai import OpenAIChatCompletionClient
model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")

In [4]:
from autogen_ext.models.ollama import OllamaChatCompletionClient
ollamamodel_client = OllamaChatCompletionClient(model="llama3.2")

### Second concept: The Message

In [4]:
from autogen_agentchat.messages import TextMessage
message = TextMessage(content="I'd like to go to Bengaluru", source="user")
message

TextMessage(source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 3, 23, 7, 25, 28, 243611, tzinfo=datetime.timezone.utc), content="I'd like to go to Bengaluru", type='TextMessage')

### Third concept: The Agent

In [6]:
from autogen_agentchat.agents import AssistantAgent

agent = AssistantAgent(name="Airline_agent", model_client=model_client, system_message="You are a helpful assistant for an airline. You give short, humorous answers.", model_client_stream=True)

### Put it all together with on_messages

In [7]:
from autogen_core import CancellationToken

response = await agent.on_messages([message], cancellation_token=CancellationToken())
response.chat_message.content

'Great choice! Bengaluru: where the traffic is almost as creative as the software! Ready to buckle up? 🛫😄'

### Let's make a local database of ticket prices

In [9]:
import os
import sqlite3

# Delete the existing database fie it if exists
if os.path.exists("tickets.db"):
    os.remove("tickets.db")

conn = sqlite3.connect("tickets.db")
c = conn.cursor()
c.execute(
    """
    CREATE TABLE cities (
          city_name TEXT PRIMARY KEY,
          round_trip_price REAL
          )
    """
)
conn.commit()
conn.close()


In [11]:
# Populate our database with some cities and prices
def save_city_price(city_name, round_trip_price):
    conn = sqlite3.connect("tickets.db")
    c = conn.cursor()
    c.execute(
        """
        REPLACE INTO cities (city_name, round_trip_price) VALUES (?,?)
        """,
        (city_name.lower(), round_trip_price)
    )

    conn.commit()
    conn.close()

# Some cities!
save_city_price("Bengaluru", 500)
save_city_price("London", 1200)
save_city_price("New York", 1500)
save_city_price("Paris", 1100)
save_city_price("Tokyo", 1300)

In [19]:
# Query function to get the price for a city
def get_city_price(city_name: str) -> float | None:
    """ Get the roundtrip ticket price to travel to the city. Returns None if the city is not found. """
    conn = sqlite3.connect("tickets.db")
    c = conn.cursor()
    c.execute(
        """
        SELECT round_trip_price FROM cities WHERE city_name = ?
        """,
        (city_name.lower(),)
    )
    result = c.fetchone()
    conn.close()
    return result[0] if result else None


In [20]:
get_city_price("Bengaluru")

500.0

In [21]:
from autogen_agentchat.agents import AssistantAgent

smart_agent = AssistantAgent(
    name="airline_agent",
    model_client=model_client,
    system_message="You are a helpful assistant for an airline. You give short, humorous answers. You have access to a database of cities and their round trip ticket prices. When the user asks for the price to a city, you use a tool to query the database and get the price.",
    model_client_stream=True,
    tools=[get_city_price],
    reflect_on_tool_use=True
)

In [23]:
response = await smart_agent.on_messages([message], cancellation_token=CancellationToken())
for inner_message in response.inner_messages:
    print(inner_message.content)

[FunctionCall(id='call_UD4IzwI8C3oDsQZ3roFsxHmq', arguments='{"city_name":"Bengaluru"}', name='get_city_price')]
[FunctionExecutionResult(content='500.0', name='get_city_price', call_id='call_UD4IzwI8C3oDsQZ3roFsxHmq', is_error=False)]


In [24]:
response.chat_message.content

'A round trip to Bengaluru is $500! Just enough to join the never-ending traffic jams and enjoy some great curry!'